# Attention — Paper-to-Code Mock (Colab)

**Paper:** Attention Is All You Need (Vaswani et al., 2017) — https://arxiv.org/abs/1706.03762

Mock (~45 min). Scope: scaled dot-product + multi-head self-attention (NOT the whole Transformer). Fill in the stub, run the content-retrieval demo, then the sanity checks. Reference solution in the last cell.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement attention (YOUR TASK)

`softmax(QK^T / sqrt(d_k)) V`, then wrap in multi-head (split -> attend -> concat -> W^O). Remember the scale, the softmax axis, and masking before softmax.

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    # TODO: scores = q @ k^T / sqrt(d_k)
    # TODO: if mask is not None: masked_fill disallowed -> -inf
    # TODO: attn = softmax(scores, last dim); return attn @ v, attn
    raise NotImplementedError("fill me in")


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads, self.d_k = d_model, n_heads, d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def _split(self, x):  # (B,T,d_model) -> (B,heads,T,d_k)
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        # TODO: project q,k,v; split into heads; attend; concat; apply wo
        # return self.wo(out), attn
        raise NotImplementedError("fill me in")

## 2. Demonstrate the benefit (content-based retrieval)
One token is flagged in feature 0; its payload is in feature 1; target = that payload. Attention must learn to point at it by content (no positional encoding needed).

In [ ]:
B, T, d = 256, 6, 16
mha = MultiHeadAttention(d_model=d, n_heads=1)
readout = nn.Linear(d, 1)
opt = torch.optim.Adam(list(mha.parameters()) + list(readout.parameters()), lr=3e-3)

def make_batch():
    x = torch.randn(B, T, d) * 0.5
    special = torch.randint(0, T, (B,))
    payload = torch.randn(B)
    x[torch.arange(B), special, 0] = 3.0     # flag
    x[torch.arange(B), special, 1] = payload # payload
    return x, payload.unsqueeze(1), special

for step in range(800):
    x, y, _ = make_batch()
    out, _ = mha(x)
    pred = readout(out.mean(dim=1))
    loss = F.mse_loss(pred, y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")

x, y, special = make_batch()
_, attn = mha(x)
avg_over_queries = attn[:, 0].mean(dim=1)
on_flag = avg_over_queries[torch.arange(B), special].mean().item()
print(f"avg attention on flagged token: {on_flag:.2f} (chance = {1/T:.2f})")

## 3. Sanity checks

In [ ]:
mha = MultiHeadAttention(d_model=32, n_heads=4)
x = torch.randn(8, 10, 32)
out, attn = mha(x)

# 1) shapes
assert out.shape == x.shape and attn.shape == (8, 4, 10, 10)

# 2) attention rows are distributions
assert torch.allclose(attn.sum(dim=-1), torch.ones(8, 4, 10), atol=1e-5)
assert (attn >= 0).all()

# 3) causal mask leaks no future
Tt = 10
causal = torch.tril(torch.ones(Tt, Tt)).view(1, 1, Tt, Tt)
_, attn_c = mha(x, mask=causal)
upper = torch.triu(torch.ones(Tt, Tt), diagonal=1).bool()
assert attn_c[..., upper].abs().max() < 1e-6

# 4) sqrt(d_k) scaling controls logit variance
for d_k in (8, 64, 512):
    q, k = torch.randn(2000, d_k), torch.randn(2000, d_k)
    raw = (q * k).sum(-1)
    print(f"d_k={d_k:3d}  std(raw)={raw.std():6.2f}  std(scaled)={(raw/math.sqrt(d_k)).std():.2f}")

# 5) permutation equivariance
mha.eval()
xs = torch.randn(1, 6, 32); perm = torch.randperm(6)
oa, _ = mha(xs); ob, _ = mha(xs[:, perm])
assert torch.allclose(oa[:, perm], ob, atol=1e-5)

# 6) gradients reach all projections
mha.train()
o, _ = mha(torch.randn(4, 7, 32)); o.sum().backward()
for name in ('wq', 'wk', 'wv', 'wo'):
    g = getattr(mha, name).weight.grad
    assert g is not None and g.abs().sum() > 0

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
def scaled_dot_product_attention_sol(q, k, v, mask=None):
    d_k = q.size(-1)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = scores.softmax(dim=-1)
    return attn @ v, attn


class MultiHeadAttentionSolution(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads, self.d_k = d_model, n_heads, d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        q, k, v = self._split(self.wq(x)), self._split(self.wk(x)), self._split(self.wv(x))
        out, attn = scaled_dot_product_attention_sol(q, k, v, mask)
        B, _, T, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.wo(out), attn